# claude

> Claude Code's models through the Agent SDK or the `claude` CLI. An agent with its own harness, not a completion endpoint.

Your tools travel in the system prompt as `<tool_call>` tags, never as MCP. That is not a fallback.
An enterprise-managed configuration forbids every dynamic MCP server, and the prompt is the one
channel no policy can close. `ClaudeChat.local` is `False`.


In [ ]:
#| default_exp claude

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import json, shutil, subprocess
from fastcore.all import store_attr, patch, ifnone
from rishi import core
from rishi.core import *

In [ ]:
#| export
_all_ = ['UsageStats', 'ChatCallback', 'run_cbs', 'resp_text', 'thought', 'Resp', 'StreamFormatter',
         'display_stream', 'truncated', 'hitl_policy', 'extract_fence', 'mk_toolspec', 'ToolCall']

In [ ]:
from fastcore.test import test_eq, test_fail

## The wire

One turn is one `query()` or one `claude -p` process. Claude Code has a real system-prompt channel,
so unlike Cursor the briefing goes there and only the conversation is rendered into the prompt.

The tools are the point. Claude Code declares a caller's tools to the model as an in-process MCP
server, and an organisation-managed configuration forbids every dynamic MCP server there is. On a
managed machine that path leaves the model with no tools at all. This backend never opens one. It
declares an empty MCP configuration on both paths, and the schemas go out as tags in the system
prompt, which `parse_tool_tags` reads back off the reply. A managed policy has nothing to refuse.

What it must *not* do is claim `--strict-mcp-config` or `strict_mcp_config=True`. That flag is
refused outright where an enterprise configuration exists ("You cannot use --strict-mcp-config when
an enterprise MCP config is present"). The obvious way to say "only my servers, please" is the one
shape a managed machine rejects. Declare nothing, claim nothing.

In [ ]:
#| export
CLAUDE_BIN = 'claude'   #: the CLI rishi drives, overridden per chat with `bin=`

# Claude Code's own aliases resolve to the latest build of each. The dated ids work too.
opus5    = 'claude-opus-5'
opus48   = 'claude-opus-4-8'
sonnet5  = 'claude-sonnet-5'
sonnet46 = 'claude-sonnet-4-6'
haiku45  = 'claude-haiku-4-5'
fable5   = 'claude-fable-5'

#: Every id above, for anything that wants to offer the list rather than reach for one of them.
CLAUDE_MODELS = {'opus5': opus5, 'opus48': opus48, 'sonnet5': sonnet5, 'sonnet46': sonnet46,
                 'haiku45': haiku45, 'fable5': fable5}

#: Claude Code's own tools this backend refuses by default. The agent gets its tools from the caller,
#: and letting the harness shell out as well is a second, ungoverned way to touch the machine.
CLAUDE_DISALLOWED = ('Bash', 'Write', 'Edit', 'NotebookEdit')

#: Managed configurations accept only an empty declared MCP server list.
NO_MCP = '{"mcpServers":{}}'

def claude_bin(bin=CLAUDE_BIN):
    "Absolute path to the `claude` binary, or a `FileNotFoundError` that says how to get one."
    if (p := shutil.which(bin)): return p
    raise FileNotFoundError(
        f'{bin!r} is not on $PATH. Install Claude Code (https://claude.com/claude-code) and run '
        f'`{bin} /login`. rishi drives it as a subprocess and never reads your credentials.')

def sdk_available():
    "Is the Claude Agent SDK importable here?"
    try:
        from claude_agent_sdk import query  # noqa: F401
        return True
    except ImportError: return False

def claude_via(via=None):
    "Which path to take: what you named, else the SDK when it is installed, else the CLI."
    if via not in (None, 'sdk', 'cli'): raise ValueError(f"via must be 'sdk', 'cli' or None, not {via!r}")
    if via == 'sdk' and not sdk_available(): raise ImportError(
        'via=\'sdk\' needs the Claude Agent SDK: pip install \'rishi[claude]\'. The CLI path needs '
        'none of it. Leave via=None and rishi takes whichever is here.')
    return via or ('sdk' if sdk_available() else 'cli')

def norm_claude_usage(u, model=None):
    "Claude Code's usage block -> rishi's, so a Claude turn adds up with a local one."
    if not u: return {}
    cache = (u.get('cache_read_input_tokens') or 0) + (u.get('cache_creation_input_tokens') or 0)
    pt, ct = (u.get('input_tokens') or 0) + cache, u.get('output_tokens') or 0
    return {'prompt_tokens': pt, 'completion_tokens': ct, 'total_tokens': pt + ct,
            'cached_tokens': u.get('cache_read_input_tokens') or 0, 'model': model}

def norm_claude(d, model=None):
    "A Claude Code result -> a rishi `Resp`, with `<tool_call>` tags read out of the text."
    if d.get('is_error'): raise RuntimeError(f"claude failed: {d.get('result') or d.get('subtype')}")
    text, th = split_think(d.get('result') or '')
    text, tcs = parse_tool_tags(text)
    res = {'role': 'assistant', 'content': text}
    if th: res['channels'] = {'thought': th}
    if tcs: res['tool_calls'] = tcs
    res['usage'] = norm_claude_usage(d.get('usage'), model)
    return Resp(res)

In [ ]:
test_eq(claude_via('cli'), 'cli')
test_fail(lambda: claude_via('rest'), contains='must be')
r = norm_claude({'result': 'ok\n<tool_call>\n{"name": "ls", "arguments": {"path": "."}}\n</tool_call>',
                 'usage': {'input_tokens': 10, 'output_tokens': 5, 'cache_read_input_tokens': 90}}, opus5)
test_eq(resp_text(r), 'ok')
test_eq(r['tool_calls'][0]['function']['name'], 'ls')
test_eq(r['usage']['total_tokens'], 105)          # cache reads are prompt tokens too
test_fail(lambda: norm_claude({'is_error': True, 'result': 'nope'}), contains='nope')

## The chat

In [ ]:
#| export
class ClaudeChat(ToolLoopMixin, Chat):
    "Chat against a Claude Code model, with the same `rishi.core.Chat` API over the Agent SDK or the CLI."
    _runtime = 'claude'
    _dflt_cbs = [UsageCallback, ToolReminderCallback, SlidingWindowCallback]
    mk_content, mk_msg, mk_msgs = staticmethod(mk_oai_content), staticmethod(mk_oai_msg), staticmethod(mk_oai_msgs)
    local = False   #: the binary is local, the model is not

    def __init__(self, model=None, *, runtime=None, model_path=None, sp='', messages=None, tools=None,
                 ctx_limit=None, approve=None, tool_max_len=None, max_steps=10, parallel_tools=False,
                 max_parallel_tools=None, final_prompt=dflt_final_prompt_,
                 permission_mode='auto',    # Claude Code's gate on its *own* tools. Yours are rishi's
                 claude_tools=None,         # Claude Code's own tools, allowlisted. None -> none at all when this chat carries tools
                 claude_disallowed=CLAUDE_DISALLOWED,   # ...and the ones it may never use
                 workspace=None,            # directory Claude Code works in. None -> the cwd
                 effort=None,               # 'low'/'medium'/'high'/'xhigh'/'max'. None -> the default
                 bare=True,                 # answer as a model: no CLAUDE.md, skills, plugins or hooks. See `_cmd`
                 via=None,                  # 'sdk' or 'cli'. None -> the SDK when it is installed
                 bin=CLAUDE_BIN, timeout=600, settings=None, cbs=None, default_cbs=True, **opts):
        self.model_id = core.split_runtime(model)[1] or opus5
        self._set_tools(tools)
        store_attr('permission_mode,claude_tools,claude_disallowed,workspace,effort,bare,bin,timeout,settings,opts')
        self.via = claude_via(via)
        self.ctx_limit, self._ctx_tokens = ctx_limit, 0
        self._setup(model=model, sp=sp, messages=messages, tools=tools, approve=approve,
                    tool_max_len=tool_max_len, max_steps=max_steps, parallel_tools=parallel_tools,
                    max_parallel_tools=max_parallel_tools, final_prompt=final_prompt, cbs=cbs,
                    default_cbs=default_cbs)

    @property
    def tool_channel(self):
        "Where this chat's tool schemas travel. Always the prompt, to get past a managed MCP policy."
        return 'tags'

    @property
    def use_sdk(self):
        "Is this chat going through the Agent SDK rather than the CLI?"
        return self.via == 'sdk'

    @property
    def token_count(self):
        "An estimate of the prompt this chat would send next. Claude Code reports no live usage."
        return est_tokens(self._prompt()) + est_tokens(self._sp())

    def _sp(self):
        "The briefing plus the tool schemas, for the one channel a managed MCP policy cannot close."
        return tag_tools_sp(self.toolspecs, self.sp)

    def _prompt(self):
        "This turn's whole conversation as text. The briefing has a channel of its own."
        return render_prompt(self.hist)

    def _note_usage(self, r):
        "Remember what the turn cost. This is billing volume, not occupancy. See `token_count`."
        self._ctx_tokens = (r.get('usage') or {}).get('total_tokens') or self._ctx_tokens
        return r

    def _recreate_conv(self):
        "Nothing to invalidate when rishi's history moves. Every turn re-sends the whole thing."
        pass

    def close(self):
        "Nothing to release. A turn is a process or a `query`, and neither outlives it."
        pass

## The CLI path

`claude -p` with `--output-format json`, and an empty `--mcp-config`. No dynamic server is declared,
so no policy has anything to refuse.

In [ ]:
#| export
@patch
def _cmd(self:ClaudeChat, fmt):
    "The `claude` command line for one turn, minus the prompt."
    cmd = [claude_bin(self.bin), '-p', '--output-format', fmt, '--model', self.model_id,
           '--mcp-config', NO_MCP]   # no `--strict-mcp-config`: see `NO_MCP`
    if fmt == 'stream-json': cmd += ['--verbose']   # `-p` refuses stream-json without it
    if self.bare: cmd += ['--safe-mode']
    if (sp := self._sp()): cmd += ['--system-prompt', sp]
    if self.permission_mode: cmd += ['--permission-mode', self.permission_mode]
    if self.effort: cmd += ['--effort', self.effort]
    if self.settings: cmd += ['--settings', str(self.settings)]
    if self.workspace: cmd += ['--add-dir', str(self.workspace)]
    if self.claude_tools: cmd += ['--allowed-tools', *self.claude_tools]
    elif self.toolspecs: cmd += ['--tools', '']
    if self.claude_disallowed: cmd += ['--disallowed-tools', *self.claude_disallowed]
    return cmd

@patch
def _run(self:ClaudeChat, fmt, prompt=None):
    "Run one turn and return the finished process. A non-zero exit is the CLI's message, not a traceback."
    r = subprocess.run(self._cmd(fmt), input=ifnone(prompt, self._prompt()), capture_output=True,
                       text=True, timeout=self.timeout, cwd=str(self.workspace) if self.workspace else None)
    if r.returncode != 0: raise RuntimeError(f'claude exited {r.returncode}: {(r.stderr or r.stdout).strip()[:400]}')
    return r

## The SDK path

`claude_agent_sdk.query` is one turn, asynchronous, yielding messages. The options carry the same
refusal to open an MCP server that `_cmd` does.

In [ ]:
#| export
@patch
def _opts(self:ClaudeChat, sp):
    "Agent SDK options for one turn, with no MCP server for a managed policy to refuse."
    from claude_agent_sdk import ClaudeAgentOptions
    kw = dict(model=self.model_id, system_prompt=sp or None,
              cwd=str(self.workspace) if self.workspace else None,
              permission_mode=self.permission_mode, settings=self.settings,
              mcp_servers={}, strict_mcp_config=False,   # see `NO_MCP`
              disallowed_tools=list(self.claude_disallowed or ()))
    if self.claude_tools is not None: kw['allowed_tools'] = list(self.claude_tools)
    elif self.toolspecs: kw['tools'] = []   # rishi's tools are the only channel - see `_cmd`
    if self.effort: kw['effort'] = self.effort
    if self.bare: kw.update(setting_sources=[], skills=[])   # the SDK's `--safe-mode`; see `_cmd`
    return ClaudeAgentOptions(**{**kw, **self.opts})

@patch
def _sdk_events(self:ClaudeChat, prompt, sp):
    "One `query` as `(kind, value)` pairs: `thought`, `text`, and one final `result` dict."
    from claude_agent_sdk import query, AssistantMessage, ResultMessage, TextBlock, ThinkingBlock
    async def _agen():
        text, res = [], None
        async for m in query(prompt=prompt, options=self._opts(sp)):
            if isinstance(m, AssistantMessage):
                for b in m.content:
                    if isinstance(b, ThinkingBlock) and b.thinking: yield 'thought', b.thinking
                    elif isinstance(b, TextBlock) and b.text: text.append(b.text); yield 'text', b.text
            elif isinstance(m, ResultMessage): res = m
        if res is None: raise RuntimeError('the Agent SDK ended without a result')
        yield 'result', {'result': res.result or ''.join(text), 'usage': res.usage, 'is_error': res.is_error, 'subtype': res.subtype}
    return sync_iter(_agen, stop=self._cancel)

## The two steps `ToolLoopMixin` drives

In [ ]:
#| export
@patch
def _model_step(self:ClaudeChat, max_output_tokens=None):
    "One wire call, through whichever path this chat uses."
    if self.use_sdk:
        out = next(v for k, v in self._sdk_events(self._prompt(), self._sp()) if k == 'result')
        return self._note_usage(norm_claude(out, self.model_id))
    return self._note_usage(norm_claude(json.loads(self._run('json').stdout), self.model_id))

@patch
def _stream_step(self:ClaudeChat, max_output_tokens=None):
    "The same turn, streamed. Thinking gets its own channel, and tag calls are never rendered as prose."
    split, out = StreamSplit(), None
    if self.use_sdk:
        for kind, v in self._sdk_events(self._prompt(), self._sp()):
            if kind == 'thought': yield {'channels': {'thought': v}}
            elif kind == 'text': yield from split.feed(v)
            else: out = v
    else:
        proc = subprocess.Popen(self._cmd('stream-json') + ['--include-partial-messages'],
                                stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
                                text=True, cwd=str(self.workspace) if self.workspace else None)
        # `with proc` waits for the CLI on the way out, without a timeout. A cancelled turn abandons
        # this generator, so kill the process rather than block the caller until it finishes alone.
        with killed_on_exit(proc):
            proc.stdin.write(self._prompt()); proc.stdin.close()   # stdin, not argv: see `_run`
            for line in proc.stdout:
                if not (line := line.strip()): continue
                try: o = json.loads(line)
                except json.JSONDecodeError: continue
                if o.get('type') == 'stream_event':
                    dl = (o.get('event') or {}).get('delta') or {}
                    if (t := dl.get('thinking')): yield {'channels': {'thought': t}}
                    elif (t := dl.get('text')): yield from split.feed(t)
                elif o.get('type') == 'result': out = o
        if out is None: raise RuntimeError(f'claude ended without a result: {proc.stderr.read()[:400]}')
    yield from split.finish()
    self._step_res = self._note_usage(norm_claude(out, self.model_id))

@patch
def _oneshot(self:ClaudeChat, prompt, sp='', think=None, max_tokens=None):
    "Stateless one-shot text, through whichever path this chat uses."
    if self.use_sdk:
        out = next(v for k, v in self._sdk_events(prompt, sp) if k == 'result')
        return resp_text(norm_claude(out, self.model_id))
    return resp_text(norm_claude(json.loads(self._run('json', prompt).stdout), self.model_id))

## Tests

Nothing here starts a model. What is asserted is the command line and the options, which is where the
enterprise contract lives and the part that fails silently if it regresses.

In [ ]:
def _fn(query: str) -> str:
    "Search the code."
    return ''

c = ClaudeChat('claude/claude-opus-5', sp='be brief', tools=[_fn], via='cli',
               messages=['what is 2 plus 2?'], claude_disallowed=('Bash',))
test_eq(c.model_id, 'claude-opus-5')          # the `claude/` prefix is stripped, the id is not
test_eq(c.local, False)

cmd = c._cmd('json')
# The enterprise contract: no dynamic server is declared, and `--strict-mcp-config` is *not* claimed.
# A managed machine refuses that flag outright, so asking for it is how this path used to fail.
test_eq(cmd[cmd.index('--mcp-config') + 1], NO_MCP)
test_eq('--strict-mcp-config' in cmd, False)
test_eq(cmd[cmd.index('--model') + 1], 'claude-opus-5')
test_eq(cmd[cmd.index('--disallowed-tools') + 1], 'Bash')
# and the prompt is not on the command line at all: `--disallowed-tools` is variadic and would eat it
test_eq(c._prompt() in cmd, False)

# ...so the schemas have to be somewhere, and the system prompt is where
sp = cmd[cmd.index('--system-prompt') + 1]
test_eq('be brief' in sp and '"_fn"' in sp and '<tool_call>' in sp, True)
# and the briefing is *not* also in the prompt, which has a channel of its own here
test_eq('be brief' not in c._prompt(), True)

# `-p --output-format stream-json` is refused outright without `--verbose`, and only then
test_eq('--verbose' in c._cmd('stream-json'), True)
test_eq('--verbose' in c._cmd('json'), False)

# and rishi's tools are the only tool channel: Claude Code's own are off while this chat carries any
test_eq(cmd[cmd.index('--tools') + 1], '')
test_eq('--tools' in ClaudeChat(via='cli')._cmd('json'), False)

# the harness's own context is off by default: this is a model, not the user's IDE agent
test_eq('--safe-mode' in cmd, True)
test_eq('--safe-mode' in ClaudeChat(via='cli', bare=False)._cmd('json'), False)
# and the schemas travel in the prompt, because a managed MCP policy refuses every native channel
test_eq(c.tool_channel, 'tags')


In [ ]:
#| eval: false
# The same contract on the SDK path. `eval: false` only because it needs the SDK installed.
o = ClaudeChat('claude/claude-opus-5', sp='be brief', tools=[_fn], via='sdk')._opts('be brief')
test_eq(o.mcp_servers, {})
test_eq(o.strict_mcp_config, False)
test_eq(o.max_turns, None)
test_eq(o.system_prompt, 'be brief')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()